### Maximal Marginal Relevance
MMR (Maximal Marginal Relevance) is a powerful diversity-aware retrieval technique used in information retrieval and RAG pipelines to balance relevance and novelty when selecting documents.

In [1]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_core.prompts import PromptTemplate
from langchain_classic.schema import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain

In [2]:
model_name = "BAAI/bge-large-en-v1.5"
model_kwargs = {"device": "cpu"}

try:
    # Load from disk only if already present
    print(f"Searching for {model_name} locally...")
    embeddings = HuggingFaceEmbeddings(
        model_name=model_name, model_kwargs={**model_kwargs, "local_files_only": True}
    )
    print("Model loaded from local storage.")

except Exception as e:
    # If not found, download it
    print("Model not found locally. Downloading from Hugging Face...")
    embeddings = HuggingFaceEmbeddings(model_name=model_name, model_kwargs=model_kwargs)
    print("Download complete and model initialized!")

Searching for BAAI/bge-large-en-v1.5 locally...
Model loaded from local storage.


In [3]:
# Load environment variables
load_dotenv()

# Initialize LLM
llm = ChatGroq(model_name="openai/gpt-oss-20b")
llm

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x00000200AA82A6C0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000200AA0670E0>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [4]:
# Step 1: Load and chunk the document
loader = TextLoader("langchain_rag_dataset.txt")
raw_docs = loader.load()
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(raw_docs)
chunks

[Document(metadata={'source': 'langchain_rag_dataset.txt'}, page_content='LangChain is an open-source framework designed to simplify the development of applications using large language models (LLMs).\nLangChain provides abstractions for working with prompts, chains, memory, and agents, making it easier to build complex LLM-based systems.'),
 Document(metadata={'source': 'langchain_rag_dataset.txt'}, page_content='The framework supports integration with various vector databases like FAISS and Chroma for semantic retrieval.\nLangChain enables Retrieval-Augmented Generation (RAG) by allowing developers to fetch relevant context before generating responses.'),
 Document(metadata={'source': 'langchain_rag_dataset.txt'}, page_content='Memory in LangChain helps models retain previous interactions, making multi-turn conversations more coherent.\nAgents in LangChain can use tools like calculators, search APIs, or custom functions based on the instructions they receive.'),
 Document(metadata={'

In [5]:
# Step 2: FAISS Vector Store with HuggingFace Embeddings
embedding_model = embeddings
vectorstore = FAISS.from_documents(chunks, embedding_model)

In [6]:
### Step 3: Create MMR Retirever
retriever = vectorstore.as_retriever(search_type="mmr", search_kwargs={"k": 3})

In [7]:
# Step 4: Prompt and LLM
prompt = PromptTemplate.from_template(
    """
Answer the question based on the context provided.

Context:
{context}

Question: {input}
"""
)

In [8]:
# Step 5: RAG Pipeline
document_chain = create_stuff_documents_chain(llm=llm, prompt=prompt)
rag_chain = create_retrieval_chain(
    retriever=retriever, combine_docs_chain=document_chain
)

In [9]:
# Step 6: Query
query = {"input": "How does LangChain support agents and memory?"}
response = rag_chain.invoke(query)

print("✅ Answer:\n", response["answer"])

✅ Answer:
 LangChain supports agents and memory by providing dedicated abstractions and tooling that let you build stateful, multi‑turn LLM applications with minimal boilerplate.

**Memory**

* **Conversation memory** – Stores past prompts, responses, and other context so that each new turn can refer back to earlier dialogue.  
* **Custom memory back‑ends** – You can plug in your own persistence layer (e.g., Redis, SQLite, vector stores) or use built‑in classes like `ConversationBufferMemory`, `ConversationSummaryMemory`, or `ConversationBufferWindowMemory`.  
* **Automatic integration** – When you create a chain or an agent, you simply pass a memory object; LangChain automatically injects the stored context into each LLM call, keeping the conversation coherent across turns.

**Agents**

* **Agent abstraction** – LangChain’s `AgentExecutor` orchestrates the decision‑making loop: the LLM decides what tool to use, the tool is executed, and the result is fed back to the LLM.  
* **Tool re